In [7]:
import pandas as pd
import numpy as np
import random 


In [ ]:
df_raw = pd.read_csv("final_surajpur_proper_reduced_2000.csv")

In [ ]:
def preprocess_gat_raj_data(df_raw:pd.DataFrame , training:bool=True)-> list:
  pred_len = 12
  obs_len = 8
  tot_len = pred_len + obs_len

  essential_columns = ['Time','Track ID','x [m]','y [m]']
  df_cleaned = df_raw[essential_columns].copy()

  column_maping = {'Time':'Frame ID','Track ID':'Agent ID','x [m]':'x','y [m]':'y'}

  df_cleaned.rename(columns=column_maping,inplace=True)
  df_cleaned.sort_values(by=['Frame ID','Agent ID'],inplace=True)
  df_group_by_frame_id = df_cleaned.groupby('Agent ID')

  all_segments = []

  for agent_id,agent_data in df_group_by_frame_id:
    frame_id = agent_data['Frame ID'].to_numpy()
    coords = agent_data[['x','y']].to_numpy()
    num_frames = len(frame_id)

    if num_frames < obs_len+1:
      continue

    for i in range (num_frames-obs_len):
      start_idx = i
      end_idx= i+obs_len

      actual_end_indx = min(i+tot_len,num_frames)

      segment_coords_abs = coords[start_idx:actual_end_indx]
      segment_frame_ids = frame_id[start_idx:actual_end_indx]

      if segment_coords_abs.shape[0] < tot_len:
        padding_needed = tot_len-segment_coords_abs.shape[0]
        padding = np.zeros((padding_needed,2))
        segment_coords_abs=np.vstack([segment_coords_abs,padding])

      # in this step we are subtracting the anchor point or the final point with all other points in the coords ok
      # in this step we have normalized the value , from the anchor point
      P_obs = segment_coords_abs[obs_len-1,:]
      P_shifted = segment_coords_abs - P_obs


      # feature calculation and augumentation

      P_displacement = P_shifted[1:,:] - P_shifted[:-1,:]
      displacement_padding = np.zeros((1,2))
      P_displacement = np.vstack([displacement_padding,P_displacement])

      # rotation for the shifted possition and the displacement

      if training:
        theta = random.uniform(0,2*np.pi)
        c,s = np.cos(theta) , np.sin(theta)
        R = np.array([[c,-s],[s,c]])

        #applying the rotation to the both the shifted and to the displacement

        P_shifted = P_shifted @ R.T
        P_displacement = P_displacement @ R.T

        # --- Final Segment Assembly ---
        all_segments.append({
                'agent_id': agent_id,
                'start_frame': frame_id[start_idx],
                'obs_coords_shifted': P_shifted[:obs_len],      # Input for GAT
                'obs_displacement': P_displacement[:obs_len],   # Input for TCN
                'pred_displacement_gt': P_displacement[obs_len:tot_len], # Ground Truth Target
                'shift_value': P_obs                            # The (X_obs, Y_obs) used for shifting # this single value used for the shifting
            })


  return all_segments


segments = preprocess_gat_raj_data(df_raw,training=True)
print(segments)


[{'agent_id': 1, 'start_frame': np.float64(0.0), 'obs_coords_shifted': array([[8.84187259, 2.12889388],
       [7.61326043, 1.82391494],
       [6.38464826, 1.518936  ],
       [5.14292087, 1.21924769],
       [3.89590285, 0.90644415],
       [2.61345144, 0.60030957],
       [1.3178848 , 0.29946562],
       [0.        , 0.        ]]), 'obs_displacement': array([[ 0.        ,  0.        ],
       [-1.22861216, -0.30497894],
       [-1.22861216, -0.30497894],
       [-1.24172739, -0.29968831],
       [-1.24701802, -0.31280354],
       [-1.28245141, -0.30613458],
       [-1.29556664, -0.30084395],
       [-1.3178848 , -0.29946562]]), 'pred_displacement_gt': array([[-1.33491233, -0.28497206],
       [-1.36505509, -0.26518788],
       [-1.38599492, -0.24149139],
       [-1.41084705, -0.20859197],
       [-1.42649625, -0.17178025],
       [-1.45134838, -0.13888084],
       [-1.46170695, -0.08895389],
       [-1.47735615, -0.05214217],
       [-1.48242409,  0.0109    ],
       [-1.48749204,  

In [12]:
import torch
from typing import List,Dict,Any,Tuple

In [ ]:
class Data_Loader:
  def __init__(self,segments:List[Dict[str,Any]],args : Any):
    self.segments = segments
    self.args = args
    self.obs_len = args.obs_len
    self.pred_len = args.pred_len
    self.total_len = args.total_len
    self.neighbour_threshold = args.neighbour_threshold

    #collecting the frame from the segments

    self.segments_by_frame = self.group_by_frame(segments)
    #getting all the frame ids

    self.all_frame_ids = list(self.segments_by_frame.keys())

  def group_by_frame(self,segments:List[Dict[str,Any]]) -> Dict[int,List[Dict[str,Any]]]:
    frame_dict = {}
    for seg in segments:
      frame = seg['start_frame']
      if frame not in frame_dict:
        frame_dict[frame] = []
      frame_dict[frame].append(seg)
    return frame_dict

  def __len__(self) ->int :
    return len(self.all_frame_ids)


  def __getitem__(self,frame_index:int) -> Tuple[torch.Tensor, ...]:
    start_frame_id = self.all_frame_ids[frame_index]
    scene_segment = self.segments_by_frame[start_frame_id]
    stacked_abs = torch.stack([torch.from_numpy(s['obs_coords_shifted'][:self.total_len]).float() for s in scene_segment],dim=1)
    stacked_disp = torch.stack([ torch.from_numpy(s['obs_displacement'][:self.total_len]).float() for s in scene_segment],dim=1)
    stacked_pred_gt = torch.stack([ torch.from_numpy(s['pred_displacement_gt'][:self.pred_len]).float()for s in scene_segment],dim=1)
    shift_value =  torch.stack([torch.from_numpy(s['shift_value']).float() for s in scene_segment],dim=0)
    # calculating the no of agent in that scene
    N = stacked_abs.shape[1]
    seq_list = torch.sum(stacked_abs,dim=2).ne(0.0).float()
    nei_lists = torch.zeros(self.obs_len,N,N,dtype=torch.float32)

    for t in range(self.obs_len):
      current_abs_coords = stacked_abs[t,:,:2]
      diff_matrix = current_abs_coords[:,None]-current_abs_coords
      dist_matrix = torch.norm(diff_matrix,dim=2)
      spatial_adj = (dist_matrix < self.neighbour_threshold).float()
      spatial_adj.fill_diagonal_(0)
      mask_t = seq_list[t,:]
      valid_agents_adj = spatial_adj * mask_t[:,None] * mask_t[None,:]
      nei_lists[t]=valid_agents_adj
    return (stacked_abs,stacked_disp,shift_value,seq_list,nei_lists,stacked_pred_gt)


  def build_scene(self,scene_segments):
   t = self.tot_len
   n = len(scene_segments)

   abs_s = torch.zeros(t,n,2)
   norm_s = torch.zeros(t,n,2)
   nei_lists = torch.zeros(self.obs_len,n,n)

   for i, seg in enumerate(scene_segments):
    P_shifted = torch.from_numpy(seg['obs_coords_shifted']).float()
    shift = torch.from_numpy(seg['shift_value']).float()
    abs_s[:,i,:] = P_shifted[:t]
    norm_s[:,i,:] = abs_s[:,i,:] - shift

   for i in range (self.obs_len):
     coords = abs_s[i]
     diff = coords[:,None] - coords[None,:]
     dist = torch.norm(diff,dim=2)
     spatial_adj = (dist < self.neighbour_threshold).float()
     spatial_adj.fill_diagonal_(0)
     nei_lists[i] = spatial_adj
   return abs_s,norm_s,nei_lists



